In [0]:
from datetime import datetime
from pyspark.sql.functions import *

In [0]:
file_path = "/Volumes/dev_catalog/landing/landing_vol/raw_roads/"
file_path2 = "/Volumes/dev_catalog/landing/landing_vol/raw_traffic/"
checkpoint_location_raw_roads = "/Volumes/dev_catalog/checkpoints/checkpoint_vol/raw_roads"
checkpoint_location_raw_roads_schema = "/Volumes/dev_catalog/checkpoints/checkpoint_vol/raw_roads_schema"

checkpoint_location_raw_traffic = "/Volumes/dev_catalog/checkpoints/checkpoint_vol/raw_traffic"
checkpoint_location_raw_traffic_schema = "/Volumes/dev_catalog/checkpoints/checkpoint_vol/raw_traffic_schema"
notebook_name = "02_load_to_bronze_batch"

In [0]:
def log_pipeline_error(step_name, error):
    spark.createDataFrame(
        [(notebook_name, step_name, str(error), datetime.now())],
        ["notebook", "step", "error_message", "error_time"]
    ).write.mode("append").saveAsTable("dev_catalog.default.pipeline_errors")

In [0]:
try:
    df = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.schemaLocation", checkpoint_location_raw_roads_schema) \
        .option("cloudFiles.allowOverwrites", "true") \
        .option("cloudFiles.schemaEvolutionMode", "rescue")\
        .load(file_path)
    df = df.withColumn("Extract_Time", current_timestamp())
  
except Exception as e:
    log_pipeline_error("read_csv_road", e)
    raise


In [0]:
df.printSchema()

In [0]:
%sql
DESCRIBE TABLE dev_catalog.bronze.raw_roads

In [0]:
try:
    df.writeStream.option("checkpointLocation", checkpoint_location_raw_roads) \
      .trigger(availableNow=True) \
      .table("dev_catalog.bronze.raw_roads")
  
except Exception as e:
    log_pipeline_error("read_csv_road", e)
    raise

In [0]:
%sql
select * from dev_catalog.bronze.raw_roads
where _rescued_data is not null;

In [0]:
try:
    df_traffic = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.schemaLocation", checkpoint_location_raw_traffic_schema) \
        .option("cloudFiles.schemaEvolutionMode", "rescue")\
        .load(file_path2)
    df_traffic = df_traffic.withColumn("Extract_Time", current_timestamp())
  
except Exception as e:
    log_pipeline_error("read_csv_traffic", e)
    raise


In [0]:
try:
    df_traffic.writeStream.option("checkpointLocation", checkpoint_location_raw_traffic) \
      .trigger(availableNow=True) \
      .table("dev_catalog.bronze.raw_traffic")
  
except Exception as e:
    log_pipeline_error("read_csv_traffic", e)
    raise


In [0]:
%sql
select count(*) from dev_catalog.bronze.raw_traffic;

In [0]:
try:
    spark.sql("""
           CREATE TABLE IF NOT EXISTS dev_catalog.bronze.vehicle_type_lookup (
            vehicle_code STRING,
            vehicle_label STRING
            );""")
    
    spark.sql("""
            INSERT INTO dev_catalog.bronze.vehicle_type_lookup VALUES
            ('EV_Car', 'Electric Car'), ('EV_Bike', 'Electric Bike'),
            ('LGV_Type', 'Light Goods Vehicle'), ('HGV_Type', 'Heavy Goods Vehicle')
        """)
except Exception as e:
    log_pipeline_error("create_table_raw_traffic",e)
    raise